# Gaussian Processes: Uncertainty Over Functions

## Historical problem

One major modelling shift in Bayesian statistics was the move from uncertainty over parameters to uncertainty over **functions**. Gaussian processes provide a clean mathematical way to express prior and posterior uncertainty over an unknown curve.

This notebook uses the classic Mauna Loa atmospheric CO$_2$ record, accessed from a bundled offline statsmodels dataset, to show prior draws and posterior function uncertainty.

In [ ]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, ExpSineSquared, RBF, WhiteKernel

ROOT = Path.cwd().resolve().parents[0]
SHARED = ROOT / "00_shared"
if str(SHARED) not in sys.path:
    sys.path.append(str(SHARED))

from plotting import save_fig, set_plot_style

set_plot_style()
rng = np.random.default_rng(404)

## Load and prepare the data

In [ ]:
co2 = sm.datasets.co2.load_pandas().data["co2"].dropna()
co2_monthly = co2.resample("MS").mean().interpolate()
co2_monthly = co2_monthly.loc["1990-01-01":"2004-12-01"]

dates = co2_monthly.index
x_year = dates.year + (dates.month - 1) / 12.0
x = (x_year - x_year.min()).to_numpy().reshape(-1, 1)
y = co2_monthly.to_numpy()

split = int(0.8 * len(x))
x_train, y_train = x[:split], y[:split]
x_test, y_test = x[split:], y[split:]

print(f"Training points: {len(x_train)}")
print(f"Test points: {len(x_test)}")

## Prior and posterior over functions

A Gaussian process is specified by a mean function and a covariance kernel. Here we use a kernel with a smooth trend, a seasonal component, and observation noise.

In [ ]:
kernel = ConstantKernel(20.0, (1e-2, 1e3)) * (
    RBF(length_scale=6.0, length_scale_bounds=(0.5, 20.0))
    + ExpSineSquared(length_scale=1.5, periodicity=1.0, periodicity_bounds=(0.8, 1.2))
) + WhiteKernel(noise_level=1.0, noise_level_bounds=(1e-3, 10.0))

gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=2, random_state=0)
gp.fit(x_train, y_train)

x_plot = np.linspace(x.min(), x.max() + 1.5, 280).reshape(-1, 1)
mean_pred, std_pred = gp.predict(x_plot, return_std=True)

prior_gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, optimizer=None, random_state=0)
prior_draws = prior_gp.sample_y(x_plot, n_samples=4, random_state=7)

print("Learned kernel:")
print(gp.kernel_)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 9))

for j in range(prior_draws.shape[1]):
    axes[0].plot(x_plot[:, 0], prior_draws[:, j], lw=1.5)
axes[0].set_title("Gaussian-process prior draws")
axes[0].set_xlabel("Years since 1990")
axes[0].set_ylabel("CO$_2$ level (arbitrary prior scale)")

axes[1].scatter(x_train[:, 0], y_train, s=12, color="#111111", alpha=0.7, label="Training data")
axes[1].scatter(x_test[:, 0], y_test, s=12, color="#54a24b", alpha=0.7, label="Held-out data")
axes[1].plot(x_plot[:, 0], mean_pred, color="#d62728", lw=2, label="Posterior mean")
axes[1].fill_between(
    x_plot[:, 0],
    mean_pred - 1.96 * std_pred,
    mean_pred + 1.96 * std_pred,
    color="#d62728",
    alpha=0.18,
    label="95% interval",
)
axes[1].set_title("Posterior function uncertainty on the CO$_2$ series")
axes[1].set_xlabel("Years since 1990")
axes[1].set_ylabel("CO$_2$ ppm")
axes[1].legend()

fig.tight_layout()
save_fig(fig, Path("figs") / "gp_prior_and_posterior.png")
plt.show()

## Interpretation

The prior draws show uncertainty over entire possible functions before seeing data. The posterior then concentrates around curves compatible with the observed CO$_2$ record, while still widening in places where extrapolation becomes less certain.

This is Bayesian modelling at the function level rather than the parameter level.

## References

- Rasmussen and Williams (2006), *Gaussian Processes for Machine Learning*.
- Standard GP regression literature built on Bayesian function estimation.